# Training remoto su Kaggle

Impostazioni: GPU T4 x2, Internet on, Persistence: Files only. Secret richiesti: S3_BUCKET, S3_ENDPOINT_URL, S3_REGION, AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY, WANDB_API_KEY (facoltativo).

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()


def secret(name, default=None):
    try:
        return secrets.get_secret(name)
    except Exception:
        return default


print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout)
print(shutil.disk_usage("/tmp"))
print(shutil.disk_usage("/kaggle/working"))

In [ ]:
WORK = Path("/kaggle/working/vehicle-detection")
REPO = "github.com/FapaKslapa/vehicle-detection.git"
if not WORK.exists():
    subprocess.run(
        ["git", "clone", "--branch", "development", f"https://{REPO}", str(WORK)],
        check=True,
    )
os.chdir(WORK)
print(subprocess.run(["git", "log", "--oneline", "-3"], capture_output=True, text=True).stdout)


In [ ]:
names = [
    "S3_BUCKET",
    "S3_ENDPOINT_URL",
    "S3_REGION",
    "AWS_ACCESS_KEY_ID",
    "AWS_SECRET_ACCESS_KEY",
    "WANDB_API_KEY",
]
lines = [f"{n}={v}" for n in names if (v := secret(n))]
lines.append("WANDB_MODE=" + ("online" if secret("WANDB_API_KEY") else "disabled"))
Path(".env").write_text("\n".join(lines) + "\n")
os.chmod(".env", 0o600)
print([line.split("=")[0] for line in lines])

In [ ]:
subprocess.run(["pip", "install", "-q", "uv", "rust-just"], check=True)

In [ ]:
base_env = {
    **os.environ,
    "UV_PROJECT_ENVIRONMENT": "/tmp/venv",
    "VIRTUAL_ENV": "/tmp/venv",
    "UV_LINK_MODE": "copy",
    "UV_SYSTEM_PYTHON": "0",
    "PATH": os.environ["PATH"] + ":/root/.local/bin",
}
subprocess.run(["just", "sync"], env=base_env, check=True)
subprocess.run(["just", "dvc-setup"], env=base_env, check=True)
subprocess.run(["uv", "run", "--no-sync", "dvc", "pull", "data/weights.dvc"], env=base_env, check=True)


## Esperimento

E3b: `SLICE_SIZE=512`, `OVERLAP=0.2`, `RUN_NAME=E3b-yolo26n-512`.
Controllo E1: `SLICE_SIZE=1024`, `OVERLAP=0`, `RUN_NAME=E1-control-yolo26n-1024`, `IMGSZ=1024`.

SEED ed EPOCHS devono coincidere con E0.

In [ ]:
RUN_NAME = "E0-yolo26n-640"
SLICE_SIZE = "1024"
OVERLAP = "0"
IMGSZ = "640"
EPOCHS = "50"
SEED = "0"

env = {
    **os.environ,
    "RUN_NAME": RUN_NAME,
    "SLICE_SIZE": SLICE_SIZE,
    "OVERLAP": OVERLAP,
    "IMGSZ": IMGSZ,
    "EPOCHS": EPOCHS,
    "SEED": SEED,
    "TILE_EXT": ".png",
    "TILES_DIR": f"/tmp/tiles-{RUN_NAME}",
    "YOLO_DIR": f"/tmp/yolo-{RUN_NAME}",
    "RESUME": str(WORK / "runs" / RUN_NAME / "weights" / "last.pt"),
    "UV_PROJECT_ENVIRONMENT": "/tmp/venv",
    "VIRTUAL_ENV": "/tmp/venv",
    "UV_LINK_MODE": "copy",
    "UV_SYSTEM_PYTHON": "0",
    "PATH": os.environ["PATH"] + ":/root/.local/bin",
}

In [ ]:
try:
    subprocess.run(["bash", "scripts/train_remote.sh"], env=env, check=True)
finally:
    Path(".env").unlink(missing_ok=True)

In [ ]:
print(Path("data/weights.dvc").read_text())
